1.1

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import jax.sharding as shd
jax.config.update('jax_num_cpu_devices', 8)

shard_map

In [ ]:
mesh = jax.make_mesh((4,2), ('x', 'y'), (shd.AxisType.Explicit, shd.AxisType.Explicit))
jax.set_mesh(mesh)

x = jnp.arange(0, 512, dtype=jnp.int32).reshape(64,8)
x = jax.device_put(x, jax.NamedSharding(mesh, jax.P('x','y')))

jax.debug.visualize_array_sharding(x)

@jax.shard_map(in_specs=jax.P('x', 'y'), out_specs=jax.P('x','y'))
def average(x):
  return x.mean().reshape(1,1)


out = average(x)

                  
  CPU 0    CPU 1  
                  
                  
  CPU 2    CPU 3  
                  
                  
  CPU 4    CPU 5  
                  
                  
  CPU 6    CPU 7  
                  

In [ ]:
out

Array([[ 61.5,  65.5],
       [189.5, 193.5],
       [317.5, 321.5],
       [445.5, 449.5]], dtype=float32)

jax.jit

In [ ]:

def average(x):
  X = mesh.shape['x']
  Y = mesh.shape['y']
  arr = jax.lax.reshape(x,(X, x.shape[0] // X, Y, x.shape[1] // Y), out_sharding=shd.NamedSharding(mesh, jax.P('x','y')))
  return arr.mean(axis=(1,3))

x = jnp.arange(0, 512, dtype=jnp.int32).reshape(64,8)

x = jax.device_put(x, jax.NamedSharding(mesh, jax.P('x','y')))

avg = jax.jit(average, out_shardings=jax.P('x','y'))

out1 = avg(x)

In [ ]:
np.testing.assert_array_equal(out, out1)

1.2

In [ ]:
mesh = jax.make_mesh((4,2), ('x', 'y'), (shd.AxisType.Explicit, shd.AxisType.Explicit))
jax.set_mesh(mesh)

x = jnp.arange(0, 512, dtype=jnp.int32).reshape(64,8)
x = jax.device_put(x, jax.NamedSharding(mesh, jax.P('x','y')))

jax.debug.visualize_array_sharding(x)
#print(x.shape)
#print(jnp.roll(x, shift=4, axis=0) - x)
@jax.shard_map(in_specs=jax.P('x', 'y'), out_specs=jax.P('x','y'))
def roll(x):
  return (jnp.roll(x, 62, axis=0) - x)

out = roll(x)

                  
  CPU 0    CPU 1  
                  
                  
  CPU 2    CPU 3  
                  
                  
  CPU 4    CPU 5  
                  
                  
  CPU 6    CPU 7  
                  

In [ ]:
out

Array([[  16,   16,   16,   16,   16,   16,   16,   16],
       [  16,   16,   16,   16,   16,   16,   16,   16],
       [  16,   16,   16,   16,   16,   16,   16,   16],
       [  16,   16,   16,   16,   16,   16,   16,   16],
       [  16,   16,   16,   16,   16,   16,   16,   16],
       [  16,   16,   16,   16,   16,   16,   16,   16],
       [  16,   16,   16,   16,   16,   16,   16,   16],
       [  16,   16,   16,   16,   16,   16,   16,   16],
       [  16,   16,   16,   16,   16,   16,   16,   16],
       [  16,   16,   16,   16,   16,   16,   16,   16],
       [  16,   16,   16,   16,   16,   16,   16,   16],
       [  16,   16,   16,   16,   16,   16,   16,   16],
       [  16,   16,   16,   16,   16,   16,   16,   16],
       [  16,   16,   16,   16,   16,   16,   16,   16],
       [-112, -112, -112, -112, -112, -112, -112, -112],
       [-112, -112, -112, -112, -112, -112, -112, -112],
       [  16,   16,   16,   16,   16,   16,   16,   16],
       [  16,   16,   16,   16,

2.1

In [ ]:
def local_moe(W,A,B):
  E,_,_ = W.shape
  S,_ = A.shape
  expert_indices = jnp.arange(E)[:,None]
  mask = B[None,:]==expert_indices # ES

  A_masked = A[None, :,:] * mask[:,:, None] # ESD

  out = A_masked @ W # ESF
  return out.sum(axis=0) #SF


In [ ]:
E = 400
S = 800
D = 400
F = 200

W = jnp.zeros((E,D,F))
A = jnp.zeros((S,D))
B = jnp.zeros((S,), dtype=jnp.int32)

jitted_local_moe = jax.jit(local_moe)

2.2

In [ ]:
import time
start_clock = time.perf_counter()

out = jitted_local_moe(W,A,B)
out.block_until_ready()
end_clock = time.perf_counter()

print(f"{(end_clock-start_clock) * 1000}ms")
print(out)

2.3.1

In [ ]:
@jax.shard_map(mesh=mesh, in_specs=((jax.P('x',None,None),jax.P('x',None),jax.P('x'))), out_specs=jax.P('x',None)) #out is S_x, D
def moe_all_gather(W,A,B):
  #all gather the tokens and routings, but keep weights sharded
  A_global = jax.lax.all_gather(A, axis_name='x', tiled='True')
  B_global = jax.lax.all_gather(B, axis_name='x', tiled='True')

  E_local, _, _ = W.shape
  #figure out what experts we have locally
  idx = jax.lax.axis_index('x')
  offset = idx * E_local
  expert_indices = jnp.arange(E_local)[:,None] + offset

  #same masking as before, here we need to mask the full data
  mask = B_global[None,:]==expert_indices
  A_masked = A_global[None, :,:] * mask[:,:, None] # ESD

  out = A_masked @ W # ESF
  out = out.sum(axis=0)

  #we have partial sums, so we need to all reduce
  out = jax.lax.psum(out, axis_name='x')

  S_local = A.shape[0]

  return jax.lax.dynamic_slice_in_dim(out,
                                      #start idx
                                      idx*S_local,
                                      #slice length
                                      S_local, axis=0) # axis 0 because we want the tokens (S) from (S,F)




2.3.2

In [ ]:
@jax.shard_map(mesh=mesh, in_specs=((jax.P('x',None,None),jax.P('x',None),jax.P('x'))), out_specs=jax.P('x',None)) #out is S_x, D
def moe_ragged(W,A,B):
  E_local, D, F = W.shape
  S_local, _ = A.shape
  num_devices = jax.lax.axis_size('x')

  target_devices = B // E_local #global token to device indices
  local_expert_idxs = B % E_local

  sort_idxs = jnp.argsort(target_devices)
  unsort_idxs = jnp.argsort(sort_idxs) #same trick as with the moe transformer

  A_sorted = A[sort_idxs]
  targets_sorted = target_devices[sort_idxs]
  experts_sorted = local_expert_idxs[sort_idxs]

  send_counts = jnp.bincount(targets_sorted, length=(num_devices)) #tokens for each device
  offsets = jnp.cumsum(send_counts) - send_counts

  #compute max amount of iterations
  global_iters = jax.lax.pmax(jnp.max(send_counts), axis_name='x') #max tokens across devices

  #stop condition
  def cond_fn(state):
    i, _ = state
    return i < global_iters

  def body_fn(state):
    i, out_buf = state

    fetch_idxs = offsets + i #moving onto next token for each device
    mask = i<send_counts #if we've moved more times than device has tokens, then output false
    safe_fetch_idxs = jnp.where(mask, fetch_idxs, S_local) #use S_local as the fallback value because it will lead to a 0.0 later

    tokens_to_send = A_sorted.at[safe_fetch_idxs].get(mode='fill', fill_value = 0.0)
    #Also need to send local experts
    experts_to_send = experts_sorted.at[safe_fetch_idxs].get(mode='fill', fill_value = 0)

    tokens_recieved = jax.lax.all_to_all(tokens_to_send, axis_name='x', split_axis=0, concat_axis=0)
    experts_recieved = jax.lax.all_to_all(experts_to_send, axis_name='x', split_axis=0, concat_axis=0)

    active_W = W[experts_recieved]

    features = jnp.einsum('nd,ndf->nf', tokens_recieved, active_W)

    returned_features = jax.lax.all_to_all(features, axis_name='x', split_axis=0, concat_axis=0)#send tokens back to original devices

    out_buf = out_buf.at[safe_fetch_idxs].set(returned_features, mode='drop') #returns back the tokens and drops the S_local values

    return (i+1, out_buf)

  out_sorted = jnp.zeros((S_local, F), dtype=A.dtype)
  init_state = (0, jax.lax.pvary(out_sorted, ('x',)))
  _, final_out = jax.lax.while_loop(cond_fn, body_fn, init_state)

  return out_sorted[unsort_idxs]

In [ ]:
import time

E = 400
S = 800
D = 400
F = 200

W = jnp.zeros((E,D,F))
A = jnp.zeros((S,D))
B = jnp.zeros((S,), dtype=jnp.int32)

start_clock = time.perf_counter()

out = moe_all_gather(W,A,B)
out.block_until_ready()

end_clock = time.perf_counter()

print(f"{(end_clock-start_clock)*1000}ms")


In [ ]:
start_clock = time.perf_counter()

out = moe_ragged(W,A,B)
out.block_until_ready()

end_clock = time.perf_counter()

print(f"{(end_clock-start_clock)*1000}ms")

2.4

In [ ]:
@jax.shard_map(mesh=mesh, in_specs=((jax.P('x',None,None),jax.P('x',None),jax.P('x'))), out_specs=jax.P('x',None)) #out is S_x, D
def moe_ragged_topk(W,A,B):
  E_local, D, F = W.shape
  S_local, _ = A.shape
  K = B.shape[1]

  num_devices = jax.lax.axis_size('x')

  A_pairs = jnp.repeat(A,K,axis=0)
  experts_global = B.reshape(-1) # remove D

  target_devices = experts_global // E_local
  local_expert_idxs = experts_global % E_local

  sort_idxs = jnp.argsort(target_devices)
  unsort_idxs = jnp.argsort(sort_idxs)

  A_sorted = A_pairs[sort_idxs]
  targets_sorted = target_devices[sort_idxs]
  experts_sorted = local_expert_idxs[sort_idxs]

  num_pairs = S_local * K

  send_counts = jnp.bincount(targets_sorted, length=num_devices)

  offsets = jnp.cumsum(send_counts) - send_counts

  global_iters = jax.lax.pmax(jnp.max(send_counts), axis_name='x') #max tokens across devices

  #stop condition
  def cond_fn(state):
    i, _ = state
    return i < global_iters

  def body_fn(state):
    i, out_buf = state

    fetch_idxs = offsets + i #moving onto next token for each device
    mask = i<send_counts #if we've moved more times than device has tokens, then output false
    safe_fetch_idxs = jnp.where(mask, fetch_idxs, num_pairs) #use num_pairs as the fallback value because it will lead to a 0.0 later

    tokens_to_send = A_sorted.at[safe_fetch_idxs].get(mode='fill', fill_value = 0.0)
    #Also need to send local experts
    experts_to_send = experts_sorted.at[safe_fetch_idxs].get(mode='fill', fill_value = 0)

    tokens_recieved = jax.lax.all_to_all(tokens_to_send, axis_name='x', split_axis=0, concat_axis=0)
    experts_recieved = jax.lax.all_to_all(experts_to_send, axis_name='x', split_axis=0, concat_axis=0)

    active_W = W[experts_recieved]

    features = jnp.einsum('nd,ndf->nf', tokens_recieved, active_W)

    returned_features = jax.lax.all_to_all(features, axis_name='x', split_axis=0, concat_axis=0)#send tokens back to original devices

    out_buf = out_buf.at[safe_fetch_idxs].set(returned_features, mode='drop') #returns back the tokens and drops the S_local values

    return (i+1, out_buf)

  out_sorted = jnp.zeros((num_pairs, F), dtype=A.dtype)
  init_state = (0, jax.lax.pvary(out_sorted, ('x',)))
  _, out_sorted = jax.lax.while_loop(cond_fn, body_fn, init_state)

  out_pairs = out_sorted[unsort_idxs]

  out = out_pairs.reshape(S_local, K, F)

  return jnp.mean(out, axis=1) #mean over experts (K) out of (S,K,F)

In [ ]:
E = 400
S = 800
D = 400
F = 200
K = 100
W = jnp.zeros((E,D,F))
A = jnp.zeros((S,D))
B = jnp.zeros((S,K), dtype=jnp.int32)


start_clock = time.perf_counter()

out = moe_ragged_topk(W,A,B)
out.block_until_ready()

end_clock = time.perf_counter()

print(f"{(end_clock-start_clock)*1000}ms")

3.1

In [ ]:
@jax.shard_map(mesh=mesh, in_specs=(jax.P('x','y'), jax.P('y', None)), out_specs=jax.P('x', None))
def naive(A_local,W_local):
  out = A_local @ W_local
  out = jax.lax.psum(out, axis_name='y') #all reduce over contracting axis 'y'
  return out

In [ ]:
import time

B = 800
D = 400
F = 200

W = jnp.zeros((D,F))
A = jnp.zeros((B,D))



start_clock = time.perf_counter()

out = naive(A,W)
out.block_until_ready()

end_clock = time.perf_counter()

print(f"{(end_clock-start_clock)*1000}ms")

In [ ]:
@jax.shard_map(mesh=mesh, in_specs=(jax.P('x','y'), jax.P('y', None)), out_specs=jax.P('x', None))
def overlapped_allreduce(A,W):
  num_tiles=4
  B_X, D_Y = A.shape
  _, F = W.shape #dont need to extract first dim of W because it is D_Y
  assert F % num_tiles == 0
  tile_size = F // num_tiles

  W_tiled = W.reshape (D_Y, num_tiles, tile_size).transpose(1,0,2) #so jax.lax.scan will iterate over num_tiles

  def f(carry, W_tile):
    out = A @ W_tile
    out = jax.lax.psum(out, axis_name='y')

    return carry, out

  _, stacked_outs = jax.lax.scan(f=f, init=None, xs=W_tiled)
  final_out = stacked_outs.transpose(1,0,2).reshape(B_X,F) #swap num_tiles with B_X, and reconstruct F

  return final_out

In [ ]:
import time

B = 800
D = 400
F = 200

W = jnp.zeros((D,F))
A = jnp.zeros((B,D))


start_clock = time.perf_counter()

out = overlapped_allreduce(A,W)
out.block_until_ready()

end_clock = time.perf_counter()

print(f"{(end_clock-start_clock)*1000}ms")

3.2

In [ ]:
@jax.shard_map(mesh=mesh, in_specs=(jax.P('x','y'), jax.P('y', None)), out_specs=jax.P('x','y'))
def overlapped_reduce_scatter(Tmp, W2):
  axis_size = jax.lax.axis_size('y')
  axis_index = jax.lax.axis_index(axis_name='y')

  B_local, F_local = Tmp.shape
  _, D = W2.shape
  D_local = D // axis_size

  W2_tiles = jnp.reshape(W2, (F_local, axis_size, D_local))

  step_0_chunk = (axis_index + axis_size - 1) % axis_size
  acc = Tmp @ W2_tiles[:, step_0_chunk, :]

  def f(acc, step):
    perm = [(i, (i+1) % axis_size) for i in range(axis_size)] #(0,1) (1,2) (2,3) (3,0)
    next_acc = jax.lax.ppermute(acc, 'y', perm=perm) #shift

    chunk_idx = (axis_index + axis_size - 1 - step) % axis_size
    local_val = Tmp @ W2_tiles[:, chunk_idx, :]

    return next_acc+local_val, None

  steps = jnp.arange(1, axis_size)
  final_acc, _ = jax.lax.scan(f, acc, steps)

  return final_acc

In [ ]:
import time

B = 800
D = 400
F = 200

W = jnp.zeros((F,D))
Tmp = jnp.zeros((B,F))


start_clock = time.perf_counter()

out = overlapped_reduce_scatter(Tmp,W)
out.block_until_ready()

end_clock = time.perf_counter()

print(f"{(end_clock-start_clock)*1000}ms")

3.3

In [ ]:
@jax.jit
def jitted_transformer_block(In, W_in, W_out):
  h = In @ W_in
  out = h @ W_out
  return out

In [ ]:
import time

B = 1024
D = 4096
F = 2048

In = jnp.zeros((B,D))
W_in = jnp.zeros((D,F))
W_out = jnp.zeros((F,D))


start_clock = time.perf_counter()

out = jitted_transformer_block(In,W_in,W_out)
out.block_until_ready()

end_clock = time.perf_counter()

print(f"{(end_clock-start_clock)*1000}ms")

In [ ]:
@jax.shard_map(mesh=mesh, in_specs=(jax.P('x','y'), jax.P('y', None), jax.P('y', None)), out_specs=jax.P('x','y'))
def sh_transformer_block(In, W_in, W_out):
  axis_size = jax.lax.axis_size('y')
  axis_index = jax.lax.axis_index(axis_name='y')

  num_tiles=4
  B_X, D_Y = In.shape
  _, F = W_in.shape
  assert F % num_tiles == 0
  tile_size = F // num_tiles

  W_tiled = W_in.reshape(D_Y, num_tiles, tile_size).transpose(1,0,2) #so jax.lax.scan will iterate over num_tiles

  def all_reduce_step(carry, W_tile):
    out = In @ W_tile
    out = jax.lax.psum(out, axis_name='y')
    return carry, out

  _, stacked_outs = jax.lax.scan(all_reduce_step, None, W_tiled)

  H_full = stacked_outs.transpose(1,0,2).reshape(B_X,F)


  F_local = F // axis_size
  start_idx = axis_index * F_local

  Tmp = jax.lax.dynamic_slice(H_full, (0, start_idx), (B_X, F_local))

  _, D = W_out.shape
  D_local = D // axis_size

  W_out_tiles = jnp.reshape(W_out, (F_local, axis_size, D_local))

  step_0_chunk = (axis_index + axis_size - 1) % axis_size
  acc = Tmp @ W_out_tiles[:, step_0_chunk, :]

  def f(acc, step):
    perm = [(i, (i+1) % axis_size) for i in range(axis_size)] #(0,1) (1,2) (2,3) (3,0)
    next_acc = jax.lax.ppermute(acc, 'y', perm=perm) #shift

    chunk_idx = (axis_index + axis_size - 1 - step) % axis_size
    local_val = Tmp @ W_out_tiles[:, chunk_idx, :]

    return next_acc+local_val, None

  steps = jnp.arange(1, axis_size)
  final_acc, _ = jax.lax.scan(f, acc, steps)

  return final_acc


In [ ]:
import time

B = 1024
D = 4096
F = 2048

In = jnp.zeros((B,D))
W_in = jnp.zeros((D,F))
W_out = jnp.zeros((F,D))


start_clock = time.perf_counter()

out = sh_transformer_block(In,W_in,W_out)
out.block_until_ready()

end_clock = time.perf_counter()

print(f"{(end_clock-start_clock)*1000}ms")

this is 1.66 times faster than a jit implementation, with B=1024, D=4096, F=2048

4.0

In [ ]:
@jax.shard_map(mesh=mesh, in_specs=(jax.P('x','y'), jax.P('y', None)), out_specs=jax.P('x','y'))
def bidirectional_reduce_scatter(Tmp, W2):
  axis_size = jax.lax.axis_size('y')
  axis_index = jax.lax.axis_index(axis_name='y')

  B_local, F_local = Tmp.shape
  _, D = W2.shape
  D_local = D // axis_size

  half_D = D_local // 2
  W2_tiles = jnp.reshape(W2, (F_local, axis_size, 2, half_D))

  cw_chunk = (axis_index + axis_size - 1) % axis_size
  ccw_chunk = (axis_index + 1) % axis_size

  acc_cw = Tmp @ W2_tiles[:, cw_chunk, 0, :]
  acc_ccw = Tmp @ W2_tiles[:, ccw_chunk, 1, :]

  def f(carry, step):
    acc_cw, acc_ccw = carry

    perm_cw = [(i, (i+1) % axis_size) for i in range(axis_size)]
    perm_ccw = [(i, (i-1) % axis_size) for i in range(axis_size)]

    next_cw = jax.lax.ppermute(acc_cw, 'y', perm=perm_cw)
    next_ccw = jax.lax.ppermute(acc_ccw, 'y', perm=perm_ccw)

    cw_idx = (axis_index + axis_size - 1 - step) % axis_size
    ccw_idx = (axis_index + 1 + step) % axis_size

    local_cw = Tmp @ W2_tiles[:, cw_idx,0, :]
    local_ccw = Tmp @ W2_tiles[:, ccw_idx,1, :]


    return (next_cw+local_cw, next_ccw+local_ccw), None

  steps = jnp.arange(1, axis_size)

  (final_cw, final_ccw), _ = jax.lax.scan(f, (acc_cw, acc_ccw), steps)

  final_acc = jnp.concatenate([final_cw, final_ccw], axis=-1)

  return final_acc

In [ ]:
import time

B = 4096
D = 8192
F = 8192

W = jnp.zeros((F,D))
Tmp = jnp.zeros((B,F))


start_clock = time.perf_counter()

out = overlapped_reduce_scatter(Tmp,W)
out.block_until_ready()

end_clock = time.perf_counter()

print(f"{(end_clock-start_clock)*1000}ms")

In [ ]:
start_clock = time.perf_counter()

out = bidirectional_reduce_scatter(Tmp,W)
out.block_until_ready()

end_clock = time.perf_counter()

print(f"{(end_clock-start_clock)*1000}ms")

bidirectional reduce scatter is 1.23 times faster than unidirectional reduce scatter

In [ ]:
@jax.shard_map(
    mesh=mesh,
    in_specs=(jax.P('x','y'), jax.P('y', None)),
    out_specs=jax.P('x', None),
    check_vma=False
)
def bidirectional_allreduce(A, W):
  axis_size = jax.lax.axis_size('y')
  B_X, D_Y = A.shape
  _, F = W.shape

  partial = A @ W
  half_F = F // 2

  forward_accum = jax.lax.slice_in_dim(partial, start_index=0, limit_index=half_F, axis=1)
  backward_accum = jax.lax.slice_in_dim(partial, start_index=half_F, limit_index=F, axis=1)

  forward_send = forward_accum
  backward_send = backward_accum

  forward_perm = [(i, (i + 1) % axis_size) for i in range(axis_size)]
  backward_perm = [(i, (i - 1) % axis_size) for i in range(axis_size)]


  def f(i, carry):
    f_acc, b_acc, f_send, b_send = carry

    next_f_send = jax.lax.ppermute(f_send, axis_name='y', perm=forward_perm)
    next_b_send = jax.lax.ppermute(b_send, axis_name='y', perm=backward_perm)

    next_f_acc = f_acc + next_f_send
    next_b_acc = b_acc + next_b_send

    return (next_f_acc, next_b_acc, next_f_send, next_b_send)

  final_f_acc, final_b_acc, _, _ = jax.lax.fori_loop(
      0,
      axis_size - 1,
      f,
      (forward_accum, backward_accum, forward_send, backward_send)
  )

  return jax.lax.concatenate([final_f_acc, final_b_acc], dimension=1)

In [ ]:
import time

B = 4096
D = 4096
F = 4096

W = jnp.zeros((D,F))
A = jnp.zeros((B,D))


start_clock = time.perf_counter()

out = overlapped_allreduce(A,W)
out.block_until_ready()

end_clock = time.perf_counter()

print(f"{(end_clock-start_clock)*1000}ms")

4672.541795000143ms


In [ ]:
start_clock = time.perf_counter()

out = bidirectional_allreduce(A,W)
out.block_until_ready()

end_clock = time.perf_counter()

print(f"{(end_clock-start_clock)*1000}ms")

3713.4702290004498ms


The bidirectional all reduce is 1.26 times faster than the unidirectional all reduce